# Initialize

In [1]:
import os, gzip
import pandas as pd
import numpy as np
import scipy.sparse as sp
from scipy.io import mmread
import scanpy as sc
import squidpy as sq
import sys
from pathlib import Path
import spatialdata as sd
from spatialdata_io import xenium
import spatialdata_plot

#Import some custom helper functions for plotting
path_skin_scripts = '/mnt/p/users/Karl/Python/Projects'
sys.path.append(path_skin_scripts)
from Skin_scripts import *

path_skin_scripts = '/mnt/p/users/Karl/Python/Projects/Human_skin_Hao'
sys.path.append(path_skin_scripts)
from Constants import *
from spatial_functions import *

import warnings
warnings.simplefilter("ignore")

/opt/miniforge3/envs/squidpy/lib/python3.12/site-packages/scanpy/_utils/__init__.py:33: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  from anndata import __version__ as anndata_version
/opt/miniforge3/envs/squidpy/lib/python3.12/site-packages/scanpy/__init__.py:24: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):
/opt/miniforge3/envs/squidpy/lib/python3.12/site-packages/scanpy/readwrite.py:16: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  if Version(anndata.__version__) >= Version("0.11.0rc2"):


# Combine original datasets

In [2]:
xenium_path = convert_path(r'P:\IMPORTANT-Backup-Sequencing-Projects\2024_Hao_HSCA_Integration_datasets\Xenium\Controls')
h5ad_path = convert_path(r'P:\IMPORTANT-Backup-Sequencing-Projects\2024_Hao_HSCA_Integration_datasets\Xenium\Processed Xenium data for HSCA label transfer')
c2l_path = convert_path(r'P:\IMPORTANT-Backup-Sequencing-Projects\2024_Hao_HSCA_Integration_datasets\cell2location\Xenium')

In [3]:
sdatas = {}
for sample in os.listdir(xenium_path):
    for subdir in os.listdir(os.path.join(xenium_path, sample)):
        for output in os.listdir(os.path.join(xenium_path, sample, subdir)):
            if 'ROI' not in output:
                continue
            roi = output.split('JF-1_')[1].split('__')[0]
            tmp_data = xenium(os.path.join(xenium_path, sample, subdir, output))
            L2 = pd.DataFrame()
            for level in os.listdir(c2l_path):
                for file in os.listdir(os.path.join(c2l_path, level)):
                    if all([file.startswith(sample), roi in file]):
                        if 'L1' in file:
                            L1 = pd.read_csv(os.path.join(c2l_path, level, file), index_col = 0)
                        else:
                            L2 = pd.concat([L2, pd.read_csv(os.path.join(c2l_path, level, file), index_col = 0)], ignore_index = True)
            replace_data = sc.read_h5ad(os.path.join(h5ad_path, sample, output, 'adata_proseg_filtered.h5ad'))
            replace_data.obs['cell_type'] = pd.NA
            shared = replace_data.obs_names.astype(str).intersection(L1.index.astype(str))
            replace_data.obs.loc[shared, 'cell_type'] = L1['cell_type']
            replace_data.obs['c2l'] = pd.NA
            shared = replace_data.obs_names.astype(str).intersection(L2.index.astype(str))
            replace_data.obs.loc[shared, 'c2l'] = L2['integrated_annotation_L2_est']
            replace_data.obs['c2l_cell_type'] = [cl.split(' - ')[0] if isinstance(cl, str) else cl for cl in replace_data.obs['c2l']] 
            sdatas[f'{sample}_{roi}'] = tmp_data
sdata = sd.concatenate(sdatas)

## Save combined

In [4]:
out_path = '/mnt/p/IMPORTANT-Backup-Sequencing-Projects/2024_Hao_HSCA_Integration_datasets/Xenium/Processed Xenium data for HSCA label transfer/'
sdata_all.write(os.path.join(out_path, 'Combined_Xenium_data.zarr'))

# Add L3 labels

In [2]:
sdata = sd.read_zarr(os.path.join(out_path, 'Combined_Xenium_data.zarr'))
sdata

SpatialData object, with associated Zarr store: /mnt/p/IMPORTANT-Backup-Sequencing-Projects/2024_Hao_HSCA_Integration_datasets/Xenium/Processed Xenium data for HSCA label transfer/Combined_Xenium_data.zarr
├── Images
│     ├── 'morphology_focus-Hip_5K_ROI_4': DataTree[cyx] (4, 23884, 19936), (4, 11942, 9968), (4, 5971, 4984), (4, 2985, 2492), (4, 1492, 1246)
│     ├── 'morphology_focus-Hip_5K_ROI_6': DataTree[cyx] (4, 23864, 22789), (4, 11932, 11394), (4, 5966, 5697), (4, 2983, 2848), (4, 1491, 1424)
│     ├── 'morphology_focus-Multiple_480_ROI_A1': DataTree[cyx] (1, 30925, 20365), (1, 15462, 10182), (1, 7731, 5091), (1, 3865, 2545), (1, 1932, 1272)
│     ├── 'morphology_focus-Multiple_480_ROI_A2': DataTree[cyx] (1, 20582, 22841), (1, 10291, 11420), (1, 5145, 5710), (1, 2572, 2855), (1, 1286, 1427)
│     ├── 'morphology_focus-Multiple_480_ROI_B1': DataTree[cyx] (1, 17254, 17320), (1, 8627, 8660), (1, 4313, 4330), (1, 2156, 2165), (1, 1078, 1082)
│     ├── 'morphology_focus-Multiple_480

In [3]:
hf_clusters = ['KC - 19: uHF', 'KC - 20: HF_1', 'KC - 21: HF_2', 'KC - 22: HF_3', 'KC - 23: HF_Anagen', 
                       'KC - 24: SG_1', 'KC - 25: SG_2', 
                       'KC - 26: Gland_Channel_1', 'KC - 27: Gland_Channel_2', 'KC - 28: Gland_Channel_3', 
                       'KC - 29: Gland_Sweat_1', 'KC - 30: Gland_Sweat_2']

In [4]:
path = '/mnt/p/IMPORTANT-Backup-Sequencing-Projects/2024_Hao_HSCA_Integration_datasets/Xenium/Xenium data from Axel with update of L3 labels/'
sites = os.listdir(path)

for site in sites:
    samples = [f for f in os.listdir(os.path.join(path, site)) if 'output' in f]
    for sample in samples:
        l3_data = sc.read_h5ad(os.path.join(path, site, sample, 'adata_proseg_filtered.h5ad'))
        table_key = f"table-{site}_{'ROI'+sample.split('ROI')[-1].split('__')[0]}"
        sdata.tables[table_key].obs['integrated_annotation_L3_est'] = l3_data.obs['integrated_annotation_L3_est'].astype(object)
        sdata.tables[table_key].obs['c2l_l2_3'] = sdata.tables[table_key].obs['c2l']
        sdata.tables[table_key].obs['c2l_l2_3'] = np.where(sdata.tables[table_key].obs['c2l_l2_3'].isin(hf_clusters), pd.NA, sdata.tables[table_key].obs['c2l_l2_3'])
        sdata.tables[table_key].obs['c2l_l2_3'] = sdata.tables[table_key].obs['c2l_l2_3'].fillna(sdata.tables[table_key].obs['integrated_annotation_L3_est'])
        #just in case replace 'other' labels, in case they remain in the annotation
        sdata.tables[table_key].obs['c2l_l2_3'] = sdata.tables[table_key].obs['c2l_l2_3'].replace('Other', pd.NA)
        sdata.tables[table_key].obs['c2l_l2_3'] = sdata.tables[table_key].obs['c2l_l2_3'].fillna(sdata.tables[table_key].obs['c2l'])

## Save L3 data

In [5]:
path = '/mnt/p/IMPORTANT-Backup-Sequencing-Projects/2024_Hao_HSCA_Integration_datasets/Xenium/Data/Xenium_all_concatenated_stereoscope_l2_3.zarr'
sdata.write(path, overwrite = True)